# Préparation des données

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from fraud_detection.data.load import PROJECT_ROOT, load_interim

FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def save(name):
    plt.savefig(FIG_DIR / f"{name}.png", dpi=120, bbox_inches="tight")

In [ ]:
df = load_interim()

# Vue d'ensemble

## Forme de la base de données

In [ ]:
print(df.shape)
print(f"{df.memory_usage(deep=True).sum() / 1e9:.2f} Go")
df["isFraud"].value_counts()

In [ ]:
df.head(30)

In [ ]:
cols = df.columns.tolist()
print(cols)

In [ ]:
import re
from collections import defaultdict

families = defaultdict(list)
for col in df.columns:
    prefix = re.match(r"[A-Za-z_]+?(?=_?\d|$)", col).group()
    families[prefix].append(col)

for prefix, cols in families.items():
    print(f"{prefix} ({len(cols)}): {cols[:5]}{' ...' if len(cols) > 5 else ''}")

## Indicateur temporel

In [ ]:
print(df["TransactionDT"].min(), df["TransactionDT"].max())
print(df["TransactionDT"].max() / 86400, "jours si l'unité est la seconde")

On commentce par faire l'hypothèse que l'unité de TransactionDT est en secondes, par rapport à un temps t que l'on ne connaît pas et qui est notre origine de référence. La première transaction démarrerait donc 1 jour après t.

In [ ]:
df["hour"] = (df["TransactionDT"] // 3600) % 24
df["hour"].value_counts().sort_index().plot(
    kind="bar", figsize=(10, 3), title="Transactions par heure"
)

On remarque ici qu'il y'a un creux dans la répartition des transactions par heure. Cela nous aide à confirmer que TransactionDT est bien un décalage temporel en secondes par rapport à un moment t choisi arbitrairement.

In [ ]:
df["day"] = df["TransactionDT"] // 86400
daily = df.groupby("day")["isFraud"].agg(["size", "mean"])

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
daily["size"].plot(ax=axes[0], title="Nombre de transactions par jour")
daily["mean"].plot(ax=axes[1], title="Taux de fraude par jour")
daily["mean"].rolling(7, center=True).mean().plot(ax=axes[1], color="red", linewidth=2)
save("temps")

# Valeurs manquantes

## Vérification générale

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
print((missing > 0.9).sum(), "colonnes à plus de 90 % de manquants")
print((missing == 0).sum(), "colonnes complètes")

groups = {"V": r"^V\d+$", "C": r"^C\d+$", "D": r"^D\d+$", "M": r"^M\d+$", "id": r"^id_"}
for name, pattern in groups.items():
    cols = df.filter(regex=pattern).columns
    print(
        f"{name}: {len(cols)} colonnes, {df[cols].isna().mean().mean():.0%} de manquants en moyenne"
    )

In [ ]:
has_identity = df.filter(regex=r"^id_").notna().any(axis=1)
df.groupby(has_identity)["isFraud"].agg(["size", "mean"])

## Vérifications du lien entre identité et type de produit

In [ ]:
pd.crosstab(df["ProductCD"], has_identity)

In [ ]:
pd.crosstab(df["ProductCD"], has_identity, values=df["isFraud"], aggfunc="mean")

## Test sur le type de carte : lien avec id et type de produit

In [ ]:
pd.crosstab(df["card4"], has_identity)

In [ ]:
pd.crosstab(df["card4"], has_identity, values=df["isFraud"], aggfunc="mean")

In [ ]:
pd.crosstab(df["card4"], df["ProductCD"], values=df["isFraud"], aggfunc="mean").round(3)

In [ ]:
pd.crosstab(df["card4"], df["ProductCD"])

## Colonnes "V"

In [ ]:
v_cols = df.filter(regex=r"^V\d+$").columns
na_counts = df[v_cols].isna().sum()
blocks = na_counts.groupby(na_counts).apply(lambda s: list(s.index))
print(len(blocks), "blocs de colonnes V partageant le même nombre de manquants")
blocks.apply(len).sort_values(ascending=False).head(10)

In [ ]:
def select_uncorrelated(cols, threshold=0.9, sample=50_000):
    corr = df[cols].sample(sample, random_state=0).corr().abs()
    kept = []
    for col in cols:
        if all(corr.loc[col, k] < threshold or pd.isna(corr.loc[col, k]) for k in kept):
            kept.append(col)
    return kept

In [ ]:
for threshold in [0.95, 0.9, 0.75]:
    kept = [c for block in blocks for c in select_uncorrelated(block, threshold)]
    print(f"Seuil {threshold} : {len(kept)} colonnes V conservées sur {len(v_cols)}")

## Colonnes avec plus de 90% de manquants

In [ ]:
for col in missing[missing > 0.9].index:
    present = df[col].notna()
    print(
        f"{col:8s} renseignée : {present.mean():5.1%} | "
        f"fraude si renseignée : {df.loc[present, 'isFraud'].mean():.3f} | "
        f"sinon : {df.loc[~present, 'isFraud'].mean():.3f}"
    )

In [ ]:
pd.crosstab(df["ProductCD"], df["dist2"].notna())

In [ ]:
pd.crosstab(
    df["ProductCD"], df["dist2"].notna(), values=df["isFraud"], aggfunc="mean"
).round(3)

In [ ]:
sub = df[has_identity]
for col in ["id_07", "id_18", "id_24"]:
    present = sub[col].notna()
    print(
        f"{col}: fraude si renseignée {sub.loc[present, 'isFraud'].mean():.3f} | "
        f"sinon {sub.loc[~present, 'isFraud'].mean():.3f}"
    )

# Montant

In [ ]:
print(df.groupby("isFraud")["TransactionAmt"].describe())

fig, ax = plt.subplots(figsize=(10, 4))
for label, sub in df.groupby("isFraud"):
    ax.hist(
        np.log1p(sub["TransactionAmt"]),
        bins=60,
        density=True,
        alpha=0.5,
        label="Fraude" if label else "Légitime",
    )
ax.set_xlabel("log(1 + montant)")
ax.legend()
save("montant")

In [ ]:
cents = (df["TransactionAmt"] * 100).round() % 100
df.groupby(cents == 0)["isFraud"].mean()

In [ ]:
amt = df["TransactionAmt"]
assert amt.dtype == "float64", "TransactionAmt doit être en float64"

n_dec = pd.Series(
    np.select(
        [(amt * 10).round(6) % 1 == 0, (amt * 100).round(6) % 1 == 0],
        [1, 2],
        default=3,
    ),
    index=df.index,
    name="n_decimales",
)

print(df.groupby(n_dec)["isFraud"].agg(["size", "mean"]))

In [ ]:
pd.crosstab(df["ProductCD"], n_dec.values)

In [ ]:
pd.crosstab(df["ProductCD"], n_dec.values)
pd.crosstab(df["ProductCD"], n_dec.values, values=df["isFraud"], aggfunc="mean").round(
    3
)

# Variables catégorielles

In [ ]:
for col in ["ProductCD", "card4", "card6", "DeviceType"]:
    print(
        df.groupby(col, observed=True)["isFraud"]
        .agg(["size", "mean"])
        .sort_values("mean", ascending=False),
        "\n",
    )

# Colonnes candidates pour le graphe

In [ ]:
for col in ["card1", "card2", "addr1", "P_emaildomain", "DeviceInfo"]:
    counts = df[col].value_counts()
    print(
        f"{col:15s} {df[col].nunique():>6} valeurs | "
        f"{df[col].isna().mean():5.1%} manquants | "
        f"taille médiane d'un groupe : {counts.median():.0f} | max : {counts.max()}"
    )

In [ ]:
fraud_per_card = df.groupby("card1")["isFraud"].transform("sum")
frauds = df["isFraud"] == 1
share = (fraud_per_card[frauds] > 1).mean()
print(f"{share:.0%} des fraudes partagent leur card1 avec au moins une autre fraude")

g = df.groupby("card1")["isFraud"].agg(["size", "mean"])
g[g["size"] >= 20]["mean"].hist(bins=50, figsize=(10, 4))
plt.title("Taux de fraude par card1 (groupes d'au moins 20 transactions)")
save("homophilie_card1")

In [ ]:
def evaluate_key(key, name):
    counts = key.value_counts()
    fraud_per_group = df.groupby(key)["isFraud"].transform("sum")
    frauds = df["isFraud"] == 1
    print(
        f"{name:12s} {key.nunique():>7} groupes | "
        f"médiane {counts.median():.0f} | max {counts.max():>6} | "
        f"{(fraud_per_group[frauds] > 1).mean():.0%} des fraudes groupées avec une autre"
    )


def make_key(*parts):
    """Concatène plusieurs colonnes en une clé texte, les manquants devenant 'NA'."""
    texts = [p.astype("string").fillna("NA") for p in parts]
    key = texts[0]
    for t in texts[1:]:
        key = key + "_" + t
    return key

In [ ]:
card_id = make_key(*(df[c] for c in ["card1", "card2", "card3", "card5", "card6"]))
uid = make_key(card_id, df["addr1"], df["day"] - df["D1"])

evaluate_key(df["card1"].astype(str), "card1")
evaluate_key(card_id, "carte")
evaluate_key(uid, "client")

In [ ]:
def shuffled_share(key, seed=0):
    shuffled = df["isFraud"].sample(frac=1, random_state=seed).values
    s = pd.Series(shuffled, index=df.index)
    fraud_per_group = s.groupby(key).transform("sum")
    return (fraud_per_group[s == 1] > 1).mean()


for name, key in [("card1", df["card1"].astype(str)), ("client", uid)]:
    print(f"{name}: {shuffled_share(key):.0%} si la fraude était répartie au hasard")

In [ ]:
uid.value_counts().head(5)

In [ ]:
df.loc[uid == uid.value_counts().index[0], "isFraud"].mean()

## Conclusion : choix des nœuds du graphe

**Aucune colonne seule n'identifie une entité.**

| Colonne | Valeurs distinctes | Plus gros groupe | Usage retenu |
|---|---|---|---|
| `P_emaildomain` | 59 | 228 355 | caractéristique |
| `card2` | 500 | 48 935 | caractéristique |
| `addr1` | 332 | 46 335 | caractéristique |
| `DeviceInfo` | 1 786 (80 % manquants) | 47 722 | caractéristique |
| `card1` | 13 553 | 14 932 | insuffisant seul |

Ces colonnes créeraient des super-nœuds reliant des dizaines de milliers de transactions sans lien réel. Combiner les cinq colonnes `card` n'apporte presque rien par rapport à `card1` seul (14 892 groupes contre 13 553).

C'est pour cela qu'on va reprendre l'**Identifiant client approximatif.** Cette technique est décrite par l'équipe gagnante de la compétition Kaggle IEEE-CIS (2019). On a `D1` qui se comporte comme un nombre de jours depuis la première transaction de la carte, et  donc d'identifier les transactions d'un même client à l'aide de `jour − D1`. La clé retenue est : carte (`card1` à `card6`) + `addr1` + (`jour − D1`).

On vérifie ce lien en mélangeant les étiquettes de fraudes, pour vérifier si le regroupement observé dépasse ce que le hasard produirait avec des groupes de cette taille (part des fraudes partageant leur groupe avec au moins une autre fraude, observée puis avec étiquettes mélangées) :

| Clé | Groupes | Médiane | Max | Observé | Au hasard |
|---|---|---|---|---|---|
| `card1` | 13 553 | 4 | 14 932 | 97 % | 90 % |
| client | 222 493 | 1 | 1 414 | 78 % | 20 % |

Ainsi, on remarque que le regroupement par `card1` est surtout mécanique, dû à la taille des groupes. A l'inverse, le regroupement par client est quatre fois supérieur au hasard : les fraudes sont réellement concentrées sur certains clients, ce qui justifie l'usage d'un graphe.

**Points importants à noter** :
- Certaines clés contiennent des valeurs manquantes (`NA`) et forment de faux groupes : elles ne créeront pas de liens.
- Le plus gros client (1 414 transactions, 0 % de fraude) impose un graphe biparti transaction–client plutôt que des liens directs entre transactions.
- D'après Vesta, les transactions ultérieures liées à une carte frauduleuse sont elles aussi étiquetées comme fraudes. Une partie du regroupement observé vient donc de cette règle, ce qui impose un split temporel, l'absence d'étiquettes de voisins en entrée du GNN, et une évaluation séparée sur les nouveaux clients.